# EstudIA · Capa 2 — Laboratorio de visión artificial con Fashion-MNIST
### De una imagen a una CNN reproducible · TensorFlow/Keras + fundamentos de MLOps

**Nivel:** básico–intermedio · **Modalidad:** Colab o Jupyter · **Proyecto:** clasificación de imágenes de prendas.

> **Objetivo de aprendizaje:** entender qué son los datos de imagen, cómo una red neuronal aprende, cómo comparar dos modelos y cómo dejar un experimento suficientemente documentado para que otra persona pueda repetirlo.

**Al terminar tendrás:** un dataset inspeccionado; un modelo base y una CNN; curvas de aprendizaje; una evaluación final sobre datos reservados; ejemplos de aciertos y errores; un modelo `.keras`; metadatos y métricas; y, opcionalmente, un experimento registrado con MLflow.

⚠️ **Importante:** Fashion-MNIST es un laboratorio didáctico, **no** un clasificador listo para fotografías reales de comercio electrónico. Las imágenes son pequeñas, centradas y en escala de grises.

**Ruta:** 0. Preparar → 1. Comprender el negocio → 2. Descargar y explorar → 3. Preparar/splitear → 4. Baseline → 5. CNN → 6. Comparar → 7. Evaluar → 8. Analizar errores → 9. Guardar y reproducir → 10. MLflow → 11. Inferencia → 12. GitHub y reto personal.

## 0. ¿Qué problema queremos resolver?

Imagina una tienda que recibe imágenes de prendas para actualizar su catálogo. Una persona revisa cada foto y elige una categoría. ¿Podría un modelo **sugerir una categoría** y dejar la decisión final a una persona cuando la predicción es dudosa?

| Elemento | Decisión de nuestro experimento |
|---|---|
| **Usuario** | Equipo que organiza un catálogo de productos |
| **Entrada** | Imagen de 28 × 28 píxeles, en escala de grises |
| **Salida** | Una de 10 categorías de prendas |
| **Línea base (baseline)** | Red neuronal densa sencilla (MLP) |
| **Modelo alternativo** | Red neuronal convolucional (CNN) |
| **Criterio de selección** | Menor pérdida de validación (`val_loss`) |
| **Evaluación final** | Accuracy, precision/recall por clase, matriz de confusión y errores en *test* |

**Pregunta para discutir:** ¿qué errores son más graves para esta tienda: confundir una camisa con una camiseta o un zapato con una camisa? ¿En qué cambiaría la evaluación?

### ¿Qué es MLOps aquí?
MLOps no significa simplemente subir un modelo a la nube. Consiste en aplicar disciplina de ingeniería para que los datos, experimentos, modelos y resultados sean **trazables, reproducibles, evaluables y reutilizables**. En este laboratorio practicaremos una versión inicial de esas ideas.

## 1. Preparar el entorno

**Opción recomendada para clase: Google Colab.** Entra a https://colab.research.google.com, selecciona **Archivo → Subir notebook** y abre este archivo `.ipynb`. Colab permite ejecutar el laboratorio desde el navegador. La CPU es suficiente; GPU puede acelerar el entrenamiento.

**Opción local:** Python compatible con TensorFlow, Jupyter/VS Code y las dependencias listadas en `requirements.txt`.

Si no tienes alguna biblioteca instalada, ejecuta una sola vez la siguiente celda **después de quitar los símbolos `#`**. En Colab, primero prueba las importaciones: instalar de nuevo bibliotecas que ya están disponibles puede provocar incompatibilidades o requerir reiniciar el entorno.

In [ ]:
# Instalación OPCIONAL — descomenta SOLO si falta alguna dependencia.
# %pip install "tensorflow>=2.16,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2" "Pillow>=10,<13"
# Para seguimiento opcional de experimentos (sección 10):
# %pip install "mlflow>=3,<4"

### 1.1. Importaciones y reproducibilidad

**¿Por qué fijamos una semilla (`seed`)?** Algunas operaciones utilizan números aleatorios (por ejemplo, inicializar los pesos o mezclar datos). Usar la misma semilla facilita comparar ejecuciones. No garantiza resultados *idénticos* en todas las computadoras: también influyen las versiones, el hardware y ciertos algoritmos no deterministas.

**Buena práctica MLOps:** registrar versiones de las dependencias y el valor de la semilla para que otro estudiante pueda intentar repetir el experimento.

In [ ]:
import os
import sys
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import sklearn
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)

SEMILLA = 42
tf.keras.utils.set_random_seed(SEMILLA)

# Modo determinista: es una ayuda a la reproducibilidad, no una garantía absoluta.
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as error:
    print('Modo determinista no activado:', error)

DIRECTORIO_ARTIFACTOS = Path('artifacts')
DIRECTORIO_MUESTRAS = Path('samples')
DIRECTORIO_ARTIFACTOS.mkdir(exist_ok=True)
DIRECTORIO_MUESTRAS.mkdir(exist_ok=True)

versiones = {
    'python': sys.version.split()[0],
    'tensorflow': tf.__version__,
    'numpy': np.__version__,
    'scikit_learn': sklearn.__version__,
    'sistema': platform.platform(),
    'semilla': SEMILLA,
}
print(json.dumps(versiones, indent=2, ensure_ascii=False))

## 2. ¿Dónde están las imágenes? Descargar Fashion-MNIST

Fashion-MNIST fue publicado por **Zalando Research** y contiene **60.000 imágenes de entrenamiento** y **10.000 imágenes de prueba**, todas de 28 × 28 píxeles y con 10 categorías etiquetadas. Esta notebook usa la copia expuesta por Keras.

**No debes descargar un ZIP para ejecutar el laboratorio.** Cuando llamas a `fashion_mnist.load_data()` por primera vez, Keras descarga los archivos necesarios y los conserva en su caché (habitualmente dentro de `~/.keras/datasets/`). Es necesario tener internet en la primera ejecución.

Si quieres revisar los archivos originales, consulta:
- Repositorio oficial: https://github.com/zalandoresearch/fashion-mnist
- Tutorial de TensorFlow: https://www.tensorflow.org/tutorials/keras/classification
- Versión en Kaggle: https://www.kaggle.com/datasets/zalando-research/fashionmnist

**¿Qué devuelve la función?** Dos pares de arreglos: `(imágenes, etiquetas)` para entrenamiento y para prueba. Cada etiqueta es un número entero del 0 al 9.

In [ ]:
(x_train_original, y_train_original), (x_test_original, y_test) = (
    tf.keras.datasets.fashion_mnist.load_data()
)

nombres_clases = [
    'Camiseta/top', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
    'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín'
]

print('Entrenamiento original:', x_train_original.shape, y_train_original.shape)
print('Prueba (reservada):', x_test_original.shape, y_test.shape)
print('Tipo de dato:', x_train_original.dtype)
print('Valores mínimo y máximo:', x_train_original.min(), x_train_original.max())
print('Primera etiqueta:', y_train_original[0], nombres_clases[int(y_train_original[0])])

### 2.1. ¿Cómo ve la computadora una imagen?

Cuando una persona observa una foto ve una prenda. Una computadora comienza viendo **números**.

- Una imagen en escala de grises de 28 × 28 es una matriz con **784 valores**.
- En este conjunto, los valores de píxel son enteros entre **0 (negro)** y **255 (blanco)**.
- El arreglo tiene forma `(alto, ancho)`, no `(ancho, alto)`.
- Una imagen a color de 28 × 28 normalmente tendría tres canales `(28, 28, 3)` para RGB; las de Fashion-MNIST tienen un único canal.

**Ejercicio en clase:** observa la imagen y una esquina de su matriz. ¿Puedes deducir qué partes tienen píxeles oscuros y claros?

In [ ]:
indice = 0
imagen = x_train_original[indice]
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
ejes[0].imshow(imagen, cmap='gray', vmin=0, vmax=255)
ejes[0].set_title(f'Clase: {nombres_clases[int(y_train_original[indice])]}')
ejes[0].axis('off')
ejes[1].imshow(imagen[:10, :10], cmap='gray', vmin=0, vmax=255)
ejes[1].set_title('Ampliación de los primeros 10 × 10 píxeles')
for i in range(10):
    for j in range(10):
        ejes[1].text(j, i, str(imagen[i, j]), ha='center', va='center', fontsize=6,
                     color='red' if imagen[i, j] > 150 else 'white')
plt.tight_layout()
plt.show()
print('Forma de esta imagen:', imagen.shape)
print('Esquina superior izquierda (5 × 5):\n', imagen[:5, :5])

### 2.2. Exploración visual y distribución de clases

**EDA** significa *Análisis Exploratorio de Datos*. Antes de entrenar, debemos verificar que podemos abrir las imágenes, que las clases son coherentes y que no encontramos problemas evidentes de calidad.

En un proyecto real también deberíamos revisar permisos de uso, imágenes repetidas, etiquetas incorrectas, desbalance de clases y diferencias entre los datos de entrenamiento y los del negocio.

In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(12, 6))
for clase, eje in enumerate(ejes.ravel()):
    idx = int(np.where(y_train_original == clase)[0][0])
    eje.imshow(x_train_original[idx], cmap='gray')
    eje.set_title(nombres_clases[clase], fontsize=10)
    eje.axis('off')
plt.suptitle('Una imagen de ejemplo por categoría')
plt.tight_layout()
plt.show()

clases, conteos = np.unique(y_train_original, return_counts=True)
plt.figure(figsize=(10, 3))
plt.bar([nombres_clases[i] for i in clases], conteos)
plt.xticks(rotation=40, ha='right')
plt.ylabel('Cantidad de ejemplos')
plt.title('Distribución de clases del entrenamiento original')
plt.tight_layout()
plt.show()

### 2.3. Guardar imágenes PNG de verdad

El dataset se descarga en un formato compacto para bibliotecas de ML; **no necesitas archivos PNG individuales para entrenar**. Sin embargo, crear un PNG por clase ayuda a comprender qué se guarda dentro de un archivo de imagen y sirve para probar inferencia después.

Este código exporta 10 ejemplos originales a `samples/`. Abre uno con cualquier visor de imágenes: ¡ahora tienes imágenes que puedes usar como archivos externos!

In [ ]:
rutas_ejemplos = []
for clase in range(10):
    idx = int(np.where(y_train_original == clase)[0][0])
    ruta = DIRECTORIO_MUESTRAS / f'fashion_clase_{clase}.png'
    Image.fromarray(x_train_original[idx], mode='L').save(ruta)
    rutas_ejemplos.append(ruta)

print(f'Se generaron {len(rutas_ejemplos)} PNG en: {DIRECTORIO_MUESTRAS.resolve()}')
print('Ejemplo:', rutas_ejemplos[0])

### 2.4. Huella digital del dataset (MLOps)

Una **huella digital (*hash*)** permite identificar si los bytes del dataset cambiaron entre ejecuciones. El hash no explica cómo son las imágenes ni garantiza que los datos estén bien etiquetados, pero sirve como evidencia de procedencia y ayuda a detectar cambios accidentales.

En un sistema de producción se registrarían versiones de datos y pipelines mediante un catálogo o herramientas como DVC o una plataforma de datos. Aquí empezamos con SHA-256 sobre los arreglos originales.

In [ ]:
import hashlib

def sha256_de_arreglo(arreglo):
    """Huella de los bytes del arreglo NumPy, en orden C."""
    return hashlib.sha256(np.ascontiguousarray(arreglo).tobytes()).hexdigest()

huella_train = sha256_de_arreglo(x_train_original)
huella_test = sha256_de_arreglo(x_test_original)
print('SHA256 train:', huella_train)
print('SHA256 test: ', huella_test)

## 3. Separar entrenamiento, validación y prueba

Este es un principio central de MLOps y de una evaluación confiable:

- **Entrenamiento (`train`)**: ejemplos utilizados para ajustar pesos.
- **Validación (`validation`)**: ejemplos usados para comparar modelos y decidir hiperparámetros; no ajustan directamente los pesos.
- **Prueba (`test`)**: ejemplos reservados para la **evaluación final** después de seleccionar la alternativa.

**Error que debemos evitar:** mirar repetidamente los resultados de test para escoger modelos. Eso contamina la evaluación final. Keras ya suministra un conjunto de prueba y lo mantendremos apartado.

Tomaremos el conjunto original de 60.000 y separaremos 10 % para validación. `stratify` mantiene aproximadamente las proporciones de clases en ambos subconjuntos.

In [ ]:
x_entrenamiento_raw, x_validacion_raw, y_entrenamiento, y_validacion = train_test_split(
    x_train_original, y_train_original,
    test_size=0.10, random_state=SEMILLA,
    stratify=y_train_original
)

print('Entrenamiento:', x_entrenamiento_raw.shape)
print('Validación:', x_validacion_raw.shape)
print('Prueba reservada:', x_test_original.shape)

assert len(x_entrenamiento_raw) == 54_000
assert len(x_validacion_raw) == 6_000
assert len(x_test_original) == 10_000
assert set(np.unique(y_entrenamiento)) == set(range(10))

### 3.1. Normalizar píxeles y añadir el canal

**¿Por qué normalizar?** Convertimos los números entre 0 y 255 a números entre 0 y 1 dividiendo por 255. Esto suele facilitar la optimización.

**¿Por qué la CNN necesita `(28, 28, 1)`?** Porque una convolución 2D recibe información de alto, ancho y número de canales; un solo canal corresponde a escala de grises.

**Buena práctica:** definir **una sola función de preprocesamiento** y reutilizarla en entrenamiento, validación, test e inferencia. Así evitamos que una imagen se prepare de una manera durante el entrenamiento y de otra durante su uso.

La división por 255 es una transformación fija: no necesita estimar estadísticas utilizando test.

In [ ]:
def preparar_imagenes(imagenes):
    """Convierte un lote (N,28,28) de píxeles 0..255 a (N,28,28,1) float32 0..1."""
    arreglo = np.asarray(imagenes)
    if arreglo.ndim != 3 or arreglo.shape[1:] != (28, 28):
        raise ValueError(f'Se espera forma (N,28,28), se recibió {arreglo.shape}')
    if not np.issubdtype(arreglo.dtype, np.number):
        raise TypeError('Los píxeles deben ser numéricos')
    if np.min(arreglo) < 0 or np.max(arreglo) > 255:
        raise ValueError('Los valores de píxel deben estar entre 0 y 255')
    return (arreglo.astype(np.float32) / 255.0)[..., np.newaxis]

x_entrenamiento = preparar_imagenes(x_entrenamiento_raw)
x_validacion = preparar_imagenes(x_validacion_raw)
# Test se prepara ahora, pero NO se usa para elegir hiperparámetros.
x_test = preparar_imagenes(x_test_original)

print('Shape para CNN:', x_entrenamiento.shape)
print('Rango normalizado:', x_entrenamiento.min(), x_entrenamiento.max())
assert x_entrenamiento.shape == (54_000, 28, 28, 1)
assert x_entrenamiento.dtype == np.float32
assert 0 <= float(x_entrenamiento.min()) <= float(x_entrenamiento.max()) <= 1

## 4. Antes de una CNN: ¿qué es una neurona y qué aprende?

Una neurona artificial recibe **entradas** (números), les asigna **pesos** (cuánto participan en la combinación), suma un **sesgo** y aplica una **función de activación**. Por ejemplo:

$$z = x_1w_1 + x_2w_2 + b \qquad y = \operatorname{ReLU}(z)$$

Si `x=[2, 1]`, `w=[0.5, -0.5]` y `b=0`, entonces `z=0.5`. ReLU deja ese valor igual. **Esto es un forward pass**: calcular la salida con los pesos actuales. **Aún no entrenamos**.

Durante el entrenamiento ocurre un ciclo:
1. **Forward pass:** obtener una predicción.
2. **Loss:** cuantificar el error usando la etiqueta correcta.
3. **Backpropagation:** calcular gradientes que indican cómo afecta cada parámetro a la pérdida.
4. **Optimizador:** ajustar los parámetros; por ejemplo, Adam.

El objetivo no es memorizar fórmulas: es comprender que el modelo **ajusta pesos a partir de ejemplos y errores**.

In [ ]:
entradas = np.array([2.0, 1.0])
pesos = np.array([0.5, -0.5])
sesgo = 0.0
z = np.dot(entradas, pesos) + sesgo
salida_relu = max(0.0, float(z))
print(f'Suma ponderada (z): {z}')
print(f'Salida con ReLU: {salida_relu}')
assert salida_relu == 0.5

## 5. Baseline: una red densa sencilla

Antes de crear una CNN, necesitamos **un punto de comparación**. Usaremos una red densa (MLP):

- `Flatten`: convierte cada imagen de 28 × 28 × 1 en una lista de 784 valores.
- `Dense(64, activation="relu")`: combina valores mediante pesos y sesgos.
- `Dense(10, activation="softmax")`: devuelve 10 puntuaciones normalizadas que suman aproximadamente 1.

**¿Qué significa `compile`?** Definimos cómo entrenar: optimizador, función de pérdida y métrica.

**¿Qué significa `fit`?** Ejecuta muchos ciclos de predicción → pérdida → gradientes → actualización.

**¿Qué es un batch?** Un grupo de ejemplos procesados juntos en un paso. **¿Qué es una epoch?** Una pasada por el conjunto de entrenamiento.

Aquí utilizamos `sparse_categorical_crossentropy` porque las etiquetas son números enteros del 0 al 9, sin *one-hot encoding*.

In [ ]:
TASA_APRENDIZAJE = 0.001
EPOCAS_MAXIMAS = 4
BATCH_SIZE = 128

def crear_mlp():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax'),
    ], name='MLP_baseline')
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=TASA_APRENDIZAJE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo

mlp = crear_mlp()
mlp.summary()

### 5.1. Entrenar el baseline

Usamos **EarlyStopping** para frenar cuando la pérdida de validación deja de mejorar; `restore_best_weights=True` recupera los mejores pesos observados. Esto ayuda a evitar entrenamientos innecesarios.

**No usamos test.** El modelo aprende con entrenamiento y lo observamos con validación.

In [ ]:
historia_mlp = mlp.fit(
    x_entrenamiento, y_entrenamiento,
    validation_data=(x_validacion, y_validacion),
    epochs=EPOCAS_MAXIMAS,
    batch_size=BATCH_SIZE,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=2, restore_best_weights=True
    )],
    verbose=2,
)

## 6. ¿Qué hace diferente a una CNN?

Una red densa ve los píxeles como una lista. Una **red convolucional (CNN)** puede aprovechar que los píxeles cercanos forman patrones locales.

- **`Conv2D`** aplica pequeños filtros (o *kernels*) que recorren la imagen. Durante el entrenamiento, los coeficientes de los filtros también se aprenden.
- **ReLU** incorpora no linealidad y mantiene los valores positivos.
- **`MaxPooling2D`** reduce la resolución de mapas de características conservando los máximos de pequeñas regiones.
- **`Flatten`** convierte los mapas de características a una lista para la clasificación final.
- **`Dense(10, softmax)`** produce una puntuación para cada tipo de prenda.

No es que la CNN tenga una regla explícita para buscar zapatos. Sus filtros se ajustan por entrenamiento para construir representaciones útiles.

**Pregunta:** ¿por qué sería interesante conservar la vecindad espacial de los píxeles, en lugar de tratarlos desde el principio como una lista?

In [ ]:
def crear_cnn():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(16, kernel_size=3, activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(32, kernel_size=3, activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10, activation='softmax'),
    ], name='CNN_FashionMNIST')
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=TASA_APRENDIZAJE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo

cnn = crear_cnn()
cnn.summary()

### 6.1. Entrenamiento de la CNN

Compararemos con el baseline **bajo las mismas condiciones de separación de datos, número máximo de épocas y batch size**. Así el experimento se puede interpretar mejor.

Durante el entrenamiento Keras muestra `loss`, `accuracy`, `val_loss` y `val_accuracy`.

**Ojo:** el modelo que tiene el accuracy de entrenamiento más alto no necesariamente es el que mejor generaliza; debemos observar validación.

In [ ]:
historia_cnn = cnn.fit(
    x_entrenamiento, y_entrenamiento,
    validation_data=(x_validacion, y_validacion),
    epochs=EPOCAS_MAXIMAS,
    batch_size=BATCH_SIZE,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=2, restore_best_weights=True
    )],
    verbose=2,
)

## 7. Comparar experimentos sin mirar test

**Regla de evaluación:** seleccionaremos el modelo con **menor pérdida de validación** y usaremos test **una sola vez para evaluar el seleccionado**. En un proyecto más grande, repetiríamos experimentos de manera controlada, registraríamos configuraciones y usaríamos validación cruzada o múltiples semillas cuando corresponda.

Esta selección también ilustra por qué debemos anotar **qué criterio decidimos antes de revisar los resultados**.

In [ ]:
resultados_validacion = {}
for nombre, modelo in [('MLP_baseline', mlp), ('CNN', cnn)]:
    perdida, exactitud = modelo.evaluate(x_validacion, y_validacion, verbose=0)
    resultados_validacion[nombre] = {
        'val_loss': float(perdida),
        'val_accuracy': float(exactitud),
        'parametros': int(modelo.count_params()),
    }

print('Resultados de VALIDACIÓN:')
print(json.dumps(resultados_validacion, indent=2, ensure_ascii=False))

nombre_elegido = min(resultados_validacion, key=lambda n: resultados_validacion[n]['val_loss'])
modelo_elegido = {'MLP_baseline': mlp, 'CNN': cnn}[nombre_elegido]
print('\nModelo seleccionado por val_loss:', nombre_elegido)

### 7.1. Visualizar las curvas de aprendizaje

Las curvas permiten responder: ¿la pérdida de entrenamiento baja? ¿Y la de validación? Si la de entrenamiento sigue bajando, pero validación empeora, podría existir **sobreajuste (overfitting)**.

Guardaremos esta figura como artefacto: un resultado que podemos compartir y comparar posteriormente, aunque se cierre la notebook.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for nombre, historia in [('MLP', historia_mlp), ('CNN', historia_cnn)]:
    ejes[0].plot(historia.history['loss'], label=f'{nombre} train')
    ejes[0].plot(historia.history['val_loss'], '--', label=f'{nombre} val')
    ejes[1].plot(historia.history['accuracy'], label=f'{nombre} train')
    ejes[1].plot(historia.history['val_accuracy'], '--', label=f'{nombre} val')
ejes[0].set(title='Pérdida', xlabel='Época', ylabel='Loss')
ejes[1].set(title='Exactitud', xlabel='Época', ylabel='Accuracy')
for eje in ejes:
    eje.legend()
    eje.grid(alpha=0.2)
plt.tight_layout()
ruta_curvas = DIRECTORIO_ARTIFACTOS / 'curvas_aprendizaje.png'
fig.savefig(ruta_curvas, dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en:', ruta_curvas)

## 8. Evaluación FINAL en el conjunto de prueba

Llegó el momento de usar el conjunto reservado: **solo después de escoger el modelo**. Este número resume su desempeño sobre las 10.000 imágenes de prueba de Fashion-MNIST, pero **no garantiza** el mismo rendimiento con fotografías de una tienda real.

Además de `accuracy`, revisaremos el **F1 macro**: promedia el F1 de cada clase y ayuda a detectar dificultades que podrían quedar escondidas en una sola cifra.

- **Precisión:** entre los ejemplos que predije como una clase, ¿cuántos lo eran?
- **Recall:** entre los ejemplos reales de una clase, ¿cuántos encontré?
- **F1:** resume precisión y recall con una media armónica.

In [ ]:
test_loss, test_accuracy = modelo_elegido.evaluate(x_test, y_test, verbose=0)
probabilidades_test = modelo_elegido.predict(x_test, verbose=0)
predicciones_test = np.argmax(probabilidades_test, axis=1)
test_f1_macro = f1_score(y_test, predicciones_test, average='macro')

metricas_finales = {
    'modelo_seleccionado': nombre_elegido,
    'criterio_seleccion': 'menor_val_loss',
    'test_loss': float(test_loss),
    'test_accuracy': float(test_accuracy),
    'test_f1_macro': float(test_f1_macro),
    'semilla': SEMILLA,
}
print(json.dumps(metricas_finales, indent=2, ensure_ascii=False))
print('\nInforme por clase:')
print(classification_report(y_test, predicciones_test,
                            labels=list(range(10)), target_names=nombres_clases,
                            digits=3, zero_division=0))

### 8.1. Matriz de confusión

Una matriz de confusión muestra **qué clases confunde el modelo**. Las filas representan la categoría real y las columnas la predicha. La diagonal principal contiene los aciertos.

**Ejercicio:** identifica las dos confusiones más frecuentes fuera de la diagonal. ¿A qué se podrían deber? Revisa las imágenes originales: *no basta con mirar los números*.

In [ ]:
matriz = confusion_matrix(y_test, predicciones_test, labels=list(range(10)))
fig, eje = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=nombres_clases).plot(
    ax=eje, cmap='Blues', xticks_rotation=45, values_format='d', colorbar=False
)
eje.set_title(f'Matriz de confusión · {nombre_elegido}')
plt.tight_layout()
ruta_matriz = DIRECTORIO_ARTIFACTOS / 'matriz_confusion.png'
fig.savefig(ruta_matriz, dpi=150, bbox_inches='tight')
plt.show()

### 8.2. Revisar errores concretos

En la práctica industrial necesitamos saber **cómo** se equivoca el modelo, no solo **cuánto**. Revisaremos seis errores y la puntuación softmax asignada a la categoría predicha.

**Aclaración:** una puntuación softmax de 0.95 **no garantiza** que un sistema esté bien calibrado ni que tenga 95 % de acierto en imágenes de ese tipo.

**Preguntas:** ¿qué clases se parecen?, ¿las imágenes están mal etiquetadas?, ¿hay una condición visual que produzca problemas repetidos?, ¿sería útil un humano en el circuito?

In [ ]:
indices_error = np.flatnonzero(predicciones_test != y_test)
print('Errores totales:', len(indices_error))

if len(indices_error) > 0:
    ejemplos = indices_error[:min(6, len(indices_error))]
    fig, ejes = plt.subplots(2, 3, figsize=(11, 7))
    for eje in ejes.ravel():
        eje.axis('off')
    for eje, idx in zip(ejes.ravel(), ejemplos):
        eje.imshow(x_test_original[idx], cmap='gray')
        verdad = nombres_clases[int(y_test[idx])]
        predicha = nombres_clases[int(predicciones_test[idx])]
        puntaje = probabilidades_test[idx, predicciones_test[idx]]
        eje.set_title(f'Real: {verdad}\nPred: {predicha} ({puntaje:.2f})', fontsize=10)
    plt.tight_layout()
    ruta_errores = DIRECTORIO_ARTIFACTOS / 'ejemplos_error.png'
    fig.savefig(ruta_errores, dpi=150, bbox_inches='tight')
    plt.show()
else:
    ruta_errores = None
    print('No se encontraron errores (algo poco usual).')

## 9. Guardar el modelo y la evidencia del experimento

MLOps también es **saber qué modelo produjo qué resultado y con qué código**. Aquí crearemos:

- `artifacts/modelo_fashion_mnist.keras`: pesos + arquitectura del modelo seleccionado.
- `artifacts/metricas.json`: métricas de evaluación, decisión y semilla.
- `artifacts/versiones.json`: dependencias y entorno.
- `artifacts/configuracion.json`: hiperparámetros y procedencia del dataset.
- `artifacts/model_card.md`: descripción breve, alcance y limitaciones.
- Gráficos de entrenamiento, errores y matriz de confusión.

**Buenas prácticas adicionales en un equipo real:** guardar identificador del commit de Git, versión/snapshot de datos, controles de acceso, revisión de sesgos, tests automáticos, registro de modelos y monitoreo de datos/predicciones tras el despliegue.

In [ ]:
ruta_modelo = DIRECTORIO_ARTIFACTOS / 'modelo_fashion_mnist.keras'
modelo_elegido.save(ruta_modelo)

configuracion = {
    'dataset': 'Zalando Research Fashion-MNIST (vía tf.keras.datasets.fashion_mnist)',
    'fuente': 'https://github.com/zalandoresearch/fashion-mnist',
    'sha256_imagenes_train_original': huella_train,
    'sha256_imagenes_test_original': huella_test,
    'train_ejemplos': int(len(x_entrenamiento)),
    'validation_ejemplos': int(len(x_validacion)),
    'test_ejemplos': int(len(x_test)),
    'seleccion': 'menor pérdida de validación',
    'epochs_max': EPOCAS_MAXIMAS,
    'batch_size': BATCH_SIZE,
    'learning_rate': TASA_APRENDIZAJE,
    'seed': SEMILLA,
    'fecha_utc': datetime.now(timezone.utc).isoformat(),
    'clases': nombres_clases,
}

for nombre, datos in [
    ('metricas.json', metricas_finales),
    ('versiones.json', versiones),
    ('configuracion.json', configuracion),
    ('comparacion_validacion.json', resultados_validacion),
]:
    (DIRECTORIO_ARTIFACTOS / nombre).write_text(
        json.dumps(datos, indent=2, ensure_ascii=False), encoding='utf-8'
    )

model_card = f"""# Model Card · Fashion-MNIST · EstudIA C2

## Propósito
Ejercicio educativo de clasificación de 10 categorías de prendas de Fashion-MNIST.

## Dataset
Zalando Research Fashion-MNIST vía Keras. Entrenamiento: {len(x_entrenamiento)};
validación: {len(x_validacion)}; test: {len(x_test)} ejemplos.

## Modelo y selección
Modelo elegido: {nombre_elegido}. Criterio: menor pérdida de validación.

## Resultado final (test)
Accuracy: {test_accuracy:.4f}. F1 macro: {test_f1_macro:.4f}.

## Entrada/salida
Entrada: arreglo float32 (28,28,1) normalizado entre 0 y 1.
Salida: 10 valores softmax. No interpretar como probabilidades calibradas.

## Limitaciones
No entrenado para fotografías reales, fondos complejos ni imágenes de mayor resolución.
No aplicar directamente a decisiones comerciales automatizadas sin validación externa.

## Reproducibilidad
Semilla: {SEMILLA}. Versiones registradas en versiones.json.
"""
(DIRECTORIO_ARTIFACTOS / 'model_card.md').write_text(model_card, encoding='utf-8')

print('Archivos guardados:')
for ruta in sorted(DIRECTORIO_ARTIFACTOS.iterdir()):
    if ruta.is_file():
        print(' •', ruta)

### 9.1. Comprobar que el modelo guardado sí funciona

**Prueba básica de reproducibilidad:** cargamos el archivo desde disco y confirmamos que sus predicciones coinciden (aproximadamente) con las del modelo en memoria sobre los mismos ejemplos.

Si un modelo no puede volver a cargarse o su entrada cambia sin aviso, tenemos un problema de ingeniería, aunque su accuracy sea alto.

In [ ]:
modelo_recuperado = tf.keras.models.load_model(ruta_modelo)
primeros_ejemplos = x_test[:5]
pred_original = modelo_elegido.predict(primeros_ejemplos, verbose=0)
pred_guardado = modelo_recuperado.predict(primeros_ejemplos, verbose=0)
np.testing.assert_allclose(pred_original, pred_guardado, rtol=1e-4, atol=1e-5)
print('✅ El modelo guardado y recuperado produce resultados equivalentes en 5 ejemplos.')

## 10. MLflow: registrar el experimento (opcional, MLOps)

Imagina que mañana cambias el número de filtros de la CNN o la tasa de aprendizaje. ¿Cómo sabrás qué configuración produjo qué resultados? **MLflow Tracking** permite organizar ejecuciones y registrar parámetros, métricas y artefactos.

Este paso es **opcional para la primera clase**. La notebook ya dejó evidencia en `artifacts/`; MLflow nos permite agruparla en un historial de experimentos.

- Instala `mlflow` si aún no está disponible: `%pip install "mlflow>=3,<4"` en una celda aparte.
- En un Jupyter local, los registros se guardarán en `./mlruns` y se pueden explorar con `mlflow ui --backend-store-uri ./mlruns` desde una terminal.
- En Colab, la interfaz gráfica requiere configuración adicional; el registro local es suficiente para demostrar el concepto, pero **los archivos pueden perderse al reiniciar el entorno**. Descárgalos o guarda una copia en Drive si deseas conservarlos.

**Importante:** utilizamos registro manual para mostrar explícitamente qué se guarda. Existe además `mlflow.tensorflow.autolog()` para automatizar buena parte del seguimiento.

In [ ]:
try:
    import mlflow
except ImportError:
    print('MLflow no está instalado. Puedes saltar este paso o ejecutar:')
    print('    %pip install "mlflow>=3,<4"')
else:
    tracking_dir = Path('mlruns')
    tracking_dir.mkdir(exist_ok=True)
    mlflow.set_tracking_uri(tracking_dir.resolve().as_uri())
    mlflow.set_experiment('EstudIA_C2_FashionMNIST')

    # Una ejecución por modelo permite comparar configuraciones en MLflow.
    experimentos = [
        ('MLP_baseline', mlp, historia_mlp),
        ('CNN', cnn, historia_cnn),
    ]
    for nombre, modelo, historia in experimentos:
        with mlflow.start_run(run_name=f'{nombre}_semilla_{SEMILLA}') as run:
            mlflow.log_params({
                'dataset': 'fashion_mnist',
                'modelo': nombre,
                'semilla': SEMILLA,
                'batch_size': BATCH_SIZE,
                'epochs_maximas': EPOCAS_MAXIMAS,
                'learning_rate': TASA_APRENDIZAJE,
                'parametros_entrenables': modelo.count_params(),
                'criterio_seleccion': 'val_loss',
            })
            mlflow.log_metrics({
                'validation_loss': resultados_validacion[nombre]['val_loss'],
                'validation_accuracy': resultados_validacion[nombre]['val_accuracy'],
            })
            for epoca, perdida in enumerate(historia.history['val_loss']):
                mlflow.log_metric('val_loss_por_epoca', float(perdida), step=epoca)
            # El conjunto test se registra SOLO para el modelo seleccionado.
            if nombre == nombre_elegido:
                mlflow.log_metrics({
                    'test_loss': float(test_loss),
                    'test_accuracy': float(test_accuracy),
                    'test_f1_macro': float(test_f1_macro),
                })
                for archivo in DIRECTORIO_ARTIFACTOS.iterdir():
                    if archivo.is_file():
                        mlflow.log_artifact(str(archivo), artifact_path='evidencias')
            print(f'✅ Run {nombre}: {run.info.run_id}')
    print('Tracking local guardado en:', tracking_dir.resolve())

## 11. Inferencia: clasificar un archivo PNG

**Inferencia** significa usar un modelo entrenado para obtener una predicción sobre una entrada.

El siguiente ejemplo abre uno de los PNG que exportamos en el paso 2.3, lo pasa a escala de grises, lo convierte a 28 × 28, normaliza los píxeles y lo pasa al modelo recuperado.

**Buena práctica MLOps:** aplicar el mismo formato/preprocesamiento que en entrenamiento. Cambiar tamaño, escala, orden de canales o rango de valores puede romper la calidad del modelo.

Puedes reemplazar la ruta por `samples/mi_prenda.png` después de colocar una imagen allí. **Precaución:** si usas una foto de celular con fondo, iluminación y escala diferentes, el resultado puede ser poco confiable por *cambio de dominio* (*domain shift*).

In [ ]:
def predecir_png(ruta_png, modelo, invertir=False):
    """Ejemplo educativo para PNG: NO garantiza buen desempeño fuera de Fashion-MNIST."""
    ruta_png = Path(ruta_png)
    if not ruta_png.is_file():
        raise FileNotFoundError(f'No existe el archivo: {ruta_png}')
    with Image.open(ruta_png) as imagen:
        imagen = imagen.convert('L')
        imagen = imagen.resize((28, 28), resample=Image.Resampling.LANCZOS)
        if invertir:
            imagen = ImageOps.invert(imagen)
        matriz = np.asarray(imagen, dtype=np.uint8)
    entrada = preparar_imagenes(matriz[np.newaxis, ...])
    scores = modelo.predict(entrada, verbose=0)[0]
    indice_prediccion = int(np.argmax(scores))
    return matriz, nombres_clases[indice_prediccion], float(scores[indice_prediccion])

ruta_ejemplo = rutas_ejemplos[0]
imagen_png, prediccion_png, score_png = predecir_png(
    ruta_ejemplo, modelo_recuperado
)
plt.imshow(imagen_png, cmap='gray')
plt.title(f'Predicción: {prediccion_png} (softmax={score_png:.2f})')
plt.axis('off')
plt.show()
print('Archivo utilizado:', ruta_ejemplo)

### 11.1. Desafío de ingeniería: ¿cómo pasarías de Fashion-MNIST a una tienda real?

Imagina fotos capturadas con teléfonos, fondos de colores, personas usando prendas, varias prendas por imagen y cambios de iluminación. Nuestro conjunto de entrenamiento **no representa esa realidad**.

Antes de automatizar un catálogo, necesitaríamos: acordar clases y casos difíciles con la empresa; obtener muestras reales con permiso; verificar etiquetas; establecer métricas por clase y umbrales de revisión humana; elegir una estrategia de transferencia de aprendizaje; evaluar con fotos nuevas del entorno real; y monitorear cambios posteriores.

**No confundas un buen resultado en un benchmark con un sistema listo para producción.**

## 12. Buenas prácticas para subir el proyecto a GitHub

Una notebook es más fácil de reutilizar cuando viene acompañada de documentación. En el archivo ZIP adjunto encontrarás la estructura recomendada:

```text
EstudIA_C2_FashionMNIST_MLOps/
├── EstudIA_C2_FashionMNIST_MLOps.ipynb
├── README.md
├── requirements.txt
└── .gitignore
```

**Qué subir:** notebook, `README.md`, `.gitignore`, instrucciones y dependencias. **Qué no subir normalmente:** entornos virtuales, pesos grandes, archivos temporales, secretos, datasets con restricciones y carpetas de experimentos locales.

Desde la terminal, dentro de la carpeta del proyecto:

```bash
git init
git add .
git commit -m "feat: baseline y CNN con Fashion-MNIST y evaluacion reproducible"
# Crear un repositorio VACÍO en GitHub antes de conectar el remoto:
git remote add origin https://github.com/TU_USUARIO/TU_REPOSITORIO.git
git branch -M main
git push -u origin main
```

**Nunca incluyas contraseñas o tokens en el repositorio.** Si trabajas en Colab, descarga la notebook y los archivos necesarios antes de cerrar la sesión.

## 13. Reto para los estudiantes: empieza tu proyecto personal

**Entrega mínima al finalizar:**

1. Notebook ejecutada desde la carga de datos hasta el resultado final (aunque el rendimiento no sea perfecto).
2. Captura o archivo de la matriz de confusión y **dos errores** explicados con palabras propias.
3. Un `README.md` que explique problema, dataset, entrada, salida, cómo ejecutar y limitaciones.
4. Un commit en GitHub que documente el primer experimento.
5. Propuesta del proyecto personal con **usuario, problema, entrada, salida, datos, métrica y primer paso realizable**.

**Retos opcionales (no cambies el test para elegir):**
- Cambiar el número de filtros de `Conv2D` y comparar con el mismo conjunto de validación.
- Entrenar 2 semillas distintas y discutir variabilidad.
- Registrar 2 runs en MLflow y explicar cuál elegirías y con qué evidencia.
- Modificar el preprocesamiento de un PNG propio y analizar el *domain shift*.

### Checklist antes de decir “terminé”

- [ ] Comprendo qué es un píxel, una entrada y un peso.
- [ ] Sé distinguir forward pass, loss, backpropagation y optimizador.
- [ ] Test no participó en la selección de modelos.
- [ ] Tengo métricas por clase y analicé errores.
- [ ] Guardé y recargué el modelo.
- [ ] Registré semilla, versiones, dataset y configuración.
- [ ] Puedo explicar las limitaciones del experimento.

## 13.1. Integración continua (CI) de nuestro repositorio

El ZIP contiene una acción sencilla de **GitHub Actions** (`.github/workflows/validar_notebook.yml`) que se ejecuta al hacer *push* o abrir un *pull request*.

Esta acción valida que la notebook sea un archivo Jupyter bien formado y que sus celdas Python tengan sintaxis válida. **No ejecuta el entrenamiento**: eso requeriría TensorFlow, las imágenes, más tiempo y pruebas de calidad de datos/modelo. La idea es introducir el hábito de detectar errores automáticamente antes de combinar cambios.

**Preguntas:** ¿qué más debería comprobar una CI de Machine Learning? Podríamos agregar pruebas del preprocesamiento, validación de esquemas, entrenamiento sobre muestras pequeñas y límites de calidad de modelos.


## 14. Apéndice docente: MNIST clásico vs. Fashion-MNIST

**MNIST clásico** contiene dígitos escritos a mano, del 0 al 9. **Fashion-MNIST** contiene prendas y usa el mismo tamaño de imagen y número de clases. Elegimos Fashion-MNIST porque coincide con el caso empresarial de clasificación de prendas de la sesión 2.

Si prefieres demostrar dígitos escritos a mano, cambia **únicamente la celda de carga de datos** (al comienzo) por:

```python
(x_train_original, y_train_original), (x_test_original, y_test) = (
    tf.keras.datasets.mnist.load_data()
)
nombres_clases = [str(i) for i in range(10)]
```

Luego vuelve a ejecutar las celdas posteriores. **No reutilices los pesos de un modelo entrenado en Fashion-MNIST** para afirmar que reconoce dígitos: cambia el conjunto de datos y vuelve a entrenar. Asimismo, cambia los nombres de archivos y experimentos para dejar clara la procedencia.

**Fuentes de estudio:**
- Fashion-MNIST oficial: https://github.com/zalandoresearch/fashion-mnist
- TensorFlow, clasificación básica: https://www.tensorflow.org/tutorials/keras/classification
- TensorFlow, API Fashion-MNIST: https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data
- TensorFlow, API MNIST clásico: https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data
- MLflow Tracking: https://mlflow.org/docs/latest/tracking/
- MLflow autolog: https://mlflow.org/docs/latest/tracking/autolog/